# 03 Matrix Factorization + Upgraded Hybrid Recommender

This notebook upgrades the movie recommender in four ways:

1. Builds a **matrix factorization** model with `TruncatedSVD`
2. Packages a reusable **HybridRecommender**
3. Combines **MF + item-item CF + genre profile + popularity**
4. Saves a deployable model artifact for the app

## 1. Imports and setup

In [ ]:
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

BASE_DIR = "/workspaces/movie-recommender-system"
PROCESSED = os.path.join(BASE_DIR, "data", "processed")
SRC_DIR = os.path.join(BASE_DIR, "src")

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from recommender import HybridRecommender, hit_rate_at_k

## 2. Load processed artifacts

In [ ]:
train = pd.read_csv(os.path.join(PROCESSED, "train.csv"))
val = pd.read_csv(os.path.join(PROCESSED, "val.csv"))
test = pd.read_csv(os.path.join(PROCESSED, "test.csv"))
movies_clean = pd.read_csv(os.path.join(PROCESSED, "movies_clean.csv"))
genre_encoded = pd.read_csv(os.path.join(PROCESSED, "genre_encoded.csv"))

print("Train:", train.shape)
print("Val  :", val.shape)
print("Test :", test.shape)
print("Movies:", movies_clean.shape)
print("Genre encoded:", genre_encoded.shape)

## 3. Fit the upgraded recommender

In [ ]:
model = HybridRecommender(
    train_df=train,
    movies_df=movies_clean,
    genre_df=genre_encoded,
    n_components=50,
    neighbor_k=15,
    min_similarity=0.10,
    min_rating_for_profile=4.0,
    cf_weight=0.30,
    mf_weight=0.45,
    genre_weight=0.15,
    popularity_weight=0.10,
).fit()

print("Model trained.")
print("Users :", len(model.user_ids))
print("Movies:", len(model.movie_ids))
print("Latent factors:", model.svd.n_components)

## 4. Matrix factorization recommendations

In [ ]:
sample_user_id = int(train["userId"].iloc[0])

mf_recs = model.recommend_mf(sample_user_id, top_n=10)
mf_recs

## 5. Upgraded hybrid recommendations

In [ ]:
hybrid_recs = model.recommend_hybrid(sample_user_id, top_n=10)
hybrid_recs

## 6. Similar movies

In [ ]:
toy_story_matches = model.find_movies_by_title("Toy Story")
toy_story_matches.head(10)

In [ ]:
toy_story_id = int(toy_story_matches.iloc[0]["movieId"])
similar_movies = model.get_similar_movies(toy_story_id, n_neighbors=10)
similar_movies

## 7. Explain a recommendation

In [ ]:
first_movie_id = int(hybrid_recs.iloc[0]["movieId"])
explanation = model.explain_recommendation(sample_user_id, first_movie_id)
explanation

## 8. Evaluate models

In [ ]:
pop_val = hit_rate_at_k(model, val, top_n=10, min_eval_rating=4.0, sample_users=100, method="popularity")
mf_val = hit_rate_at_k(model, val, top_n=10, min_eval_rating=4.0, sample_users=100, method="mf")
hybrid_val = hit_rate_at_k(model, val, top_n=10, min_eval_rating=4.0, sample_users=100, method="hybrid")

pop_test = hit_rate_at_k(model, test, top_n=10, min_eval_rating=4.0, sample_users=100, method="popularity")
mf_test = hit_rate_at_k(model, test, top_n=10, min_eval_rating=4.0, sample_users=100, method="mf")
hybrid_test = hit_rate_at_k(model, test, top_n=10, min_eval_rating=4.0, sample_users=100, method="hybrid")

results = pd.DataFrame({
    "model": ["Popularity", "Matrix Factorization", "Upgraded Hybrid"],
    "val_hit_rate@10": [pop_val, mf_val, hybrid_val],
    "test_hit_rate@10": [pop_test, mf_test, hybrid_test],
})
results

## 9. Save model artifact for the Streamlit app

In [ ]:
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

model_path = os.path.join(MODELS_DIR, "hybrid_recommender.pkl")
model.save(model_path)

print("Saved:", model_path)

## 10. Save optional matrices and report tables

In [ ]:
model.pred_df.to_csv(os.path.join(PROCESSED, "mf_prediction_matrix.csv"))
model.popularity_df.to_csv(os.path.join(PROCESSED, "popularity_df.csv"), index=False)
results.to_csv(os.path.join(PROCESSED, "model_comparison_results.csv"), index=False)

print("Saved processed outputs.")